In [0]:
from pyspark.sql.functions import col, when, rand, lit

current = spark.table("nyc311_project.gold.dim_complaint_status").filter(col("is_current") == True)

# Simulate: take complaints that are currently "Open" or "In Progress",
# and pretend 30% of them have now moved to "Closed" (a realistic next-day change)
changeable = current.filter(col("status").isin("Open", "In Progress", "Assigned", "Pending", "Started"))

simulated_changes = (changeable
    .sample(fraction=0.3, seed=42)
    .withColumn("status", lit("Closed"))
)

print(f"Simulating {simulated_changes.count()} status changes out of {changeable.count()} open/in-progress complaints")
display(simulated_changes.select("unique_key", "status").limit(10))

Simulating 892 status changes out of 2954 open/in-progress complaints


unique_key,status
61664866,Closed
61654509,Closed
61655240,Closed
61659267,Closed
61664766,Closed
61658342,Closed
61660367,Closed
61661359,Closed
61658346,Closed
61659515,Closed


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date, lit, current_timestamp

target = DeltaTable.forName(spark, "nyc311_project.gold.dim_complaint_status")

# New incoming rows, prepared with fresh SCD2 metadata
incoming = (simulated_changes
    .withColumn("valid_from", current_date())
    .withColumn("valid_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn("_scd_loaded_at", current_timestamp())
)

# Step A: close out old current rows where the status actually changed
(target.alias("t")
    .merge(
        incoming.alias("s"),
        "t.unique_key = s.unique_key AND t.is_current = true"
    )
    .whenMatchedUpdate(set={
        "valid_to": "current_date()",
        "is_current": "false"
    })
    .execute())

# Step B: insert the new current rows
incoming.write.format("delta").mode("append") \
    .saveAsTable("nyc311_project.gold.dim_complaint_status")

print("SCD2 merge complete.")

SCD2 merge complete.


In [0]:
sample_keys = [row.unique_key for row in simulated_changes.select("unique_key").limit(5).collect()]

display(spark.table("nyc311_project.gold.dim_complaint_status")
        .filter(col("unique_key").isin(sample_keys))
        .orderBy("unique_key", "valid_from"))

unique_key,status,agency,complaint_type,created_ts,valid_from,valid_to,is_current,_scd_loaded_at
61655274,Closed,DPR,Illegal Tree Damage,2024-06-30T23:31:02.000Z,2026-10-06,null,true,2026-10-06T06:14:40.303Z
61655274,In Progress,DPR,Illegal Tree Damage,2024-06-30T23:31:02.000Z,2026-10-06,null,true,2026-10-06T06:07:44.499Z
61658316,Closed,DPR,Damaged Tree,2024-06-30T20:35:13.000Z,2026-10-06,null,true,2026-10-06T06:14:40.303Z
61658316,In Progress,DPR,Damaged Tree,2024-06-30T20:35:13.000Z,2026-10-06,null,true,2026-10-06T06:07:44.499Z
61658383,Closed,DPR,Overgrown Tree/branches,2024-06-30T21:22:11.000Z,2026-10-06,null,true,2026-10-06T06:14:40.303Z
61658383,In Progress,DPR,Overgrown Tree/branches,2024-06-30T21:22:11.000Z,2026-10-06,null,true,2026-10-06T06:07:44.499Z
61659400,Closed,DPR,Overgrown Tree/branches,2024-06-30T20:39:06.000Z,2026-10-06,null,true,2026-10-06T06:14:40.303Z
61659400,In Progress,DPR,Overgrown Tree/branches,2024-06-30T20:39:06.000Z,2026-10-06,null,true,2026-10-06T06:07:44.499Z
61660266,Pending,DOT,Street Condition,2024-06-30T20:26:12.000Z,2026-10-06,null,true,2026-10-06T06:07:44.499Z
61660266,Closed,DOT,Street Condition,2024-06-30T20:26:12.000Z,2026-10-06,null,true,2026-10-06T06:14:40.303Z


In [0]:
from pyspark.sql.functions import col, lit, current_date, current_timestamp
from delta.tables import DeltaTable

# Step 1: materialize the simulated changes as an independent staging table
current = spark.table("nyc311_project.gold.dim_complaint_status").filter(col("is_current") == True)
changeable = current.filter(col("status").isin("Open", "In Progress", "Assigned", "Pending", "Started"))
simulated_changes = changeable.sample(fraction=0.3, seed=42).withColumn("status", lit("Closed"))

incoming = (simulated_changes
    .withColumn("valid_from", current_date())
    .withColumn("valid_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn("_scd_loaded_at", current_timestamp())
)

# Materialize it as a real table — breaks the lazy self-reference
incoming.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc311_project.quarantine.staging_status_changes")

incoming_materialized = spark.table("nyc311_project.quarantine.staging_status_changes")
print(f"Staged {incoming_materialized.count()} changes")

Staged 625 changes


In [0]:
# Step 2: now merge FROM the materialized staging table, not the live chain
target = DeltaTable.forName(spark, "nyc311_project.gold.dim_complaint_status")

(target.alias("t")
    .merge(
        incoming_materialized.alias("s"),
        "t.unique_key = s.unique_key AND t.is_current = true"
    )
    .whenMatchedUpdate(set={
        "valid_to": "current_date()",
        "is_current": "false"
    })
    .execute())

incoming_materialized.write.format("delta").mode("append") \
    .saveAsTable("nyc311_project.gold.dim_complaint_status")

print("SCD2 merge complete.")

SCD2 merge complete.


In [0]:
spark.sql("""
    DELETE FROM nyc311_project.gold.dim_complaint_status
    WHERE _scd_loaded_at = (
        SELECT MAX(_scd_loaded_at) FROM nyc311_project.gold.dim_complaint_status
    )
""")
print("Cleaned up bad merge attempt.")

Cleaned up bad merge attempt.


In [0]:
sample_keys = [row.unique_key for row in simulated_changes.select("unique_key").limit(5).collect()]

display(spark.table("nyc311_project.gold.dim_complaint_status")
        .filter(col("unique_key").isin(sample_keys))
        .orderBy("unique_key", "valid_from"))

unique_key,status,agency,complaint_type,created_ts,valid_from,valid_to,is_current,_scd_loaded_at
61656261,In Progress,DPR,Damaged Tree,2024-06-30T18:57:23.000Z,2026-10-06,null,true,2026-10-06T06:07:44.499Z
61656601,Open,DEP,Sewer,2024-06-30T22:41:00.000Z,2026-10-06,null,true,2026-10-06T06:07:44.499Z
61657337,In Progress,DPR,Damaged Tree,2024-06-30T19:22:35.000Z,2026-10-06,null,true,2026-10-06T06:07:44.499Z
61657343,In Progress,DPR,Damaged Tree,2024-06-30T20:35:23.000Z,2026-10-06,null,true,2026-10-06T06:07:44.499Z
61659353,In Progress,DPR,Damaged Tree,2024-06-30T19:08:30.000Z,2026-10-06,null,true,2026-10-06T06:07:44.499Z


In [0]:
from pyspark.sql.functions import col, lit, current_date, current_timestamp
from delta.tables import DeltaTable

# STEP 0 — reset: remove any bad data from earlier attempts, confirm clean baseline
spark.sql("""
    DELETE FROM nyc311_project.gold.dim_complaint_status
    WHERE is_current = false OR valid_to IS NOT NULL
""")
baseline_count = spark.table("nyc311_project.gold.dim_complaint_status").count()
print(f"Baseline row count (should be 199053, all current): {baseline_count}")

# STEP 1 — pick a FIXED, explicit set of 5 keys to track (not a random sample we might lose track of)
current = spark.table("nyc311_project.gold.dim_complaint_status").filter(col("is_current") == True)
changeable = current.filter(col("status").isin("Open", "In Progress", "Assigned", "Pending", "Started"))

tracked_keys = [row.unique_key for row in changeable.select("unique_key").limit(5).collect()]
print(f"Tracking these 5 keys through the merge: {tracked_keys}")

# STEP 2 — build the incoming changes, but ONLY for our tracked keys (small, easy to verify)
incoming = (changeable
    .filter(col("unique_key").isin(tracked_keys))
    .withColumn("status", lit("Closed"))
    .withColumn("valid_from", current_date())
    .withColumn("valid_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn("_scd_loaded_at", current_timestamp())
)

# Materialize as a real table (breaks lazy self-reference)
incoming.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc311_project.quarantine.staging_status_changes")
incoming_materialized = spark.table("nyc311_project.quarantine.staging_status_changes")
print(f"Staged {incoming_materialized.count()} rows (should be 5)")

# STEP 3 — merge: close out the old rows for these 5 keys
target = DeltaTable.forName(spark, "nyc311_project.gold.dim_complaint_status")
(target.alias("t")
    .merge(incoming_materialized.alias("s"), "t.unique_key = s.unique_key AND t.is_current = true")
    .whenMatchedUpdate(set={"valid_to": "current_date()", "is_current": "false"})
    .execute())

# STEP 4 — insert the new current rows
incoming_materialized.write.format("delta").mode("append") \
    .saveAsTable("nyc311_project.gold.dim_complaint_status")

# STEP 5 — verify using the SAME tracked_keys list from Step 1
print("\n--- VERIFICATION ---")
display(spark.table("nyc311_project.gold.dim_complaint_status")
        .filter(col("unique_key").isin(tracked_keys))
        .orderBy("unique_key", "valid_from"))

Baseline row count (should be 199053, all current): 197536
Tracking these 5 keys through the merge: ['61656601', '61659333', '61657343', '61657314', '61657337']
Staged 5 rows (should be 5)

--- VERIFICATION ---


unique_key,status,agency,complaint_type,created_ts,valid_from,valid_to,is_current,_scd_loaded_at
61656601,Closed,DEP,Sewer,2024-06-30T22:41:00.000Z,2026-10-06,null,true,2026-10-06T09:20:16.622Z
61656601,Open,DEP,Sewer,2024-06-30T22:41:00.000Z,2026-10-06,2026-10-06,false,2026-10-06T06:07:44.499Z
61657314,Closed,DPR,Damaged Tree,2024-06-30T19:38:03.000Z,2026-10-06,null,true,2026-10-06T09:20:16.622Z
61657314,In Progress,DPR,Damaged Tree,2024-06-30T19:38:03.000Z,2026-10-06,2026-10-06,false,2026-10-06T06:07:44.499Z
61657337,Closed,DPR,Damaged Tree,2024-06-30T19:22:35.000Z,2026-10-06,null,true,2026-10-06T09:20:16.622Z
61657337,In Progress,DPR,Damaged Tree,2024-06-30T19:22:35.000Z,2026-10-06,2026-10-06,false,2026-10-06T06:07:44.499Z
61657343,Closed,DPR,Damaged Tree,2024-06-30T20:35:23.000Z,2026-10-06,null,true,2026-10-06T09:20:16.622Z
61657343,In Progress,DPR,Damaged Tree,2024-06-30T20:35:23.000Z,2026-10-06,2026-10-06,false,2026-10-06T06:07:44.499Z
61659333,In Progress,DPR,Damaged Tree,2024-06-30T21:00:13.000Z,2026-10-06,2026-10-06,false,2026-10-06T06:07:44.499Z
61659333,Closed,DPR,Damaged Tree,2024-06-30T21:00:13.000Z,2026-10-06,null,true,2026-10-06T09:20:16.622Z


In [0]:
from pyspark.sql.functions import col, current_date, lit, current_timestamp

silver = spark.table("nyc311_project.silver.service_requests")

dim_complaint_status = (silver
    .select("unique_key", "status", "agency", "complaint_type", "created_ts")
    .withColumn("valid_from", current_date())
    .withColumn("valid_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn("_scd_loaded_at", current_timestamp())
)

dim_complaint_status.write.format("delta").mode("overwrite") \
    .saveAsTable("nyc311_project.gold.dim_complaint_status")

print(f"Fresh rebuild: {dim_complaint_status.count()} rows, all current")

Fresh rebuild: 199053 rows, all current
